In [1]:
import random
import time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.ndimage import gaussian_filter
from skimage.transform import resize
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

In [2]:
# frame-wise spatial baseline only - no temporal context, no respiratory-phase
# conditioning, no segmentation inputs, no neighbouring-frame or pca features
# val/test are the two subjects (besides tv1, which stays in training) whose all six
# slices survive training_exclude - see the split cell below for tv7's exclusion
VAL_SUBJECT = "TV4_20220204_sorted_UCL"
TEST_SUBJECT = "TV8_20220307_sorted_UCL"  # internal synthetic test set, not the real iqt lr-hr evaluation

CROP_SIZE = 80
SCALE_FACTOR = 1.5  # hr/lr spacing ratio baked into the export - fixed, has physical meaning

# sigma derived from the downsampling scale rather than picked by hand: the blur's
# frequency cutoff should match the resolution actually being simulated, so synthetic lr
# loses exactly the detail a genuinely coarser scan would miss. the export's fixed
# lr_synth_frames were built at sigma=1.0 (data_prep.ipynb step 8's default) which no
# longer matches sigma_ref below - that array is a reference at a different blur level
# than the training path, not a like-for-like comparison
FWHM_PX = SCALE_FACTOR
SIGMA_REF = FWHM_PX / (2 * np.sqrt(2 * np.log(2)))
SIGMA_MIN = SIGMA_REF * 0.7
SIGMA_MAX = SIGMA_REF * 1.3

CHANNELS = [16, 32, 64, 128]  # unet encoder channel counts, one per downsampling level

EPOCHS = 40
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
SEED = 0
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS = 0  
MAX_EPOCH_MINUTES = 15  # rough per-epoch budget for reasoning about a run on a shared queue

# resolves whether the notebook is run from development/ (local convention) or the repo
# root (as could happen on a remote machine), instead of assuming one hardcoded relative path
candidate_output_dirs = [Path.cwd(), Path.cwd() / "Development"]
OUTPUT_DIR = next((d / "Outputs" for d in candidate_output_dirs if (d / "Outputs").exists()), Path.cwd() / "Outputs")
EXPORT_PATH = OUTPUT_DIR / "tv_hr_crop_synthetic_lr_export.npz"
assert EXPORT_PATH.exists(), f"export file not found at {EXPORT_PATH.resolve()} - run data_prep.ipynb's export step first"

CHECKPOINT_PATH = OUTPUT_DIR / "spatial_baseline_best.pt"
HISTORY_PATH = OUTPUT_DIR / "spatial_baseline_history.csv"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)

In [3]:
# just checks 
print("device:", DEVICE)
if torch.cuda.is_available():
    print("gpu name:", torch.cuda.get_device_name(0))
    print("cuda version:", torch.version.cuda)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"total vram: {vram_gb:.1f} gb")
else:
    print("no gpu available, running on cpu")


device: cpu
no gpu available, running on cpu


In [4]:
export_data = np.load(EXPORT_PATH, allow_pickle=True)
print("export arrays:", list(export_data.files))

hr_frames = export_data["hr_frames"]
included = export_data["included"]
subject = export_data["subject"]
sl = export_data["slice"]
dyn = export_data["dynamic"]
phase_bin = export_data["phase_bin"]

print("total exported frames:", len(hr_frames))
print("frame shape:", hr_frames.shape[1:])

# lr_synth_frames is a fixed-sigma reference artefact from the export, not the training
# input - training lr is regenerated per-frame below with a sampled sigma instead

hr_frames = hr_frames[included]
subject = subject[included]
sl = sl[included]
dyn = dyn[included]
phase_bin = phase_bin[included]

print("included frame count after filtering:", len(hr_frames))


export arrays: ['hr_frames', 'lr_synth_frames', 'subject', 'slice', 'dynamic', 'phase_bin', 'phase_direction', 'anchor_row', 'anchor_col', 'crop_origin_row', 'crop_origin_col', 'included']


total exported frames: 9120
frame shape: (80, 80)
included frame count after filtering: 7030


In [5]:
# no "echo" field exists in the export at all - by construction (data_prep.ipynb step 8)
# every exported frame was pulled with echo_num=1, so te1 is the only echo present
assert "echo" not in export_data.files
echoes_used = ["TE1"]
print("echoes present in export metadata: none recorded (te1-only by construction)")
print("echo(es) used for this baseline:", echoes_used)


echoes present in export metadata: none recorded (te1-only by construction)
echo(es) used for this baseline: ['TE1']


In [6]:
all_subjects = sorted(set(subject.tolist()))
train_subjects = [s for s in all_subjects if s not in (VAL_SUBJECT, TEST_SUBJECT)]

# split by subject, never by frame - adjacent dynamics are highly correlated, a random
# frame split would leak information between train/val/test. 8 subjects total makes this
# a small-sample split
train_mask = np.isin(subject, train_subjects)
val_mask = subject == VAL_SUBJECT
heldout_mask = subject == TEST_SUBJECT

train_idx = np.where(train_mask)[0]
val_idx = np.where(val_mask)[0]
heldout_idx = np.where(heldout_mask)[0]

print("train subjects:", train_subjects)
print("val subject:", VAL_SUBJECT)
print("internal synthetic test subject:", TEST_SUBJECT)
print("train/val/test frame counts:", len(train_idx), len(val_idx), len(heldout_idx))

print("per-subject frame counts:")
for s in all_subjects:
    group = "val" if s == VAL_SUBJECT else "test" if s == TEST_SUBJECT else "train"
    print(f"  {s} ({group}): {int((subject == s).sum())}")


train subjects: ['TV1_20220222_sorted_UCL', 'TV2_20220222_sorted_UCL', 'TV3_20220208_sorted_UCL', 'TV5_20220307_sorted_UCL', 'TV6_20220307_sorted_UCL', 'TV7_20220315_sorted_UCL']
val subject: TV4_20220204_sorted_UCL
internal synthetic test subject: TV8_20220307_sorted_UCL
train/val/test frame counts: 4750 1140 1140
per-subject frame counts:
  TV1_20220222_sorted_UCL (train): 1140
  TV2_20220222_sorted_UCL (train): 760
  TV3_20220208_sorted_UCL (train): 950
  TV4_20220204_sorted_UCL (val): 1140
  TV5_20220307_sorted_UCL (train): 760
  TV6_20220307_sorted_UCL (train): 570
  TV7_20220315_sorted_UCL (train): 570
  TV8_20220307_sorted_UCL (test): 1140


In [7]:
# copied directly from data_prep.ipynb step 8 (not imported) so the forward model is
# fully visible here. mode="constant" zero-pads the blur - a genuinely coarser scan has
# no signal beyond the frame, unlike scipy's reflect default which invents mirrored
# neighbours. zero-padding means edge pixels average in fewer real neighbours and come
# out attenuated, so the blur is divided by a blurred all-ones support of the identical
# shape/sigma/mode: interior support blurs back to 1.0 (no change), edges fall below 1.0
# and get rescaled up to their true local mean
def degrade_hr_to_lr(image, blur_sigma=SIGMA_REF, downsample_factor=SCALE_FACTOR):
    blur_mode = "constant"
    blurred = gaussian_filter(image.astype(np.float64), sigma=blur_sigma, mode=blur_mode, cval=0.0)
    support = gaussian_filter(np.ones_like(image, dtype=np.float64), sigma=blur_sigma, mode=blur_mode, cval=0.0)
    blurred = blurred / np.maximum(support, 1e-3)

    size = image.shape[0]
    small_size = max(1, int(round(size / downsample_factor)))
    downsampled = resize(blurred, (small_size, small_size), order=3, mode="edge",
                          anti_aliasing=False, preserve_range=True)
    upsampled = resize(downsampled, (size, size), order=3, mode="edge",
                        anti_aliasing=False, preserve_range=True)
    return upsampled


In [8]:
def compute_percentile_scale(frames, subject, sl, low=1, high=99):
    # one (lo, hi) pair per subject/slice time series, not per frame - keeps genuine
    # temporal intensity changes across dynamics instead of erasing them
    scale = {}
    for key in set(zip(subject.tolist(), sl.tolist())):
        series = frames[(subject == key[0]) & (sl == key[1])]
        scale[key] = tuple(np.percentile(series, [low, high]))
    return scale


def normalise(img, lo, hi):
    return np.clip((img - lo) / max(hi - lo, 1e-6), 0.0, 1.0)


def denormalise(img, lo, hi):
    # maps a normalised prediction back to the original hr intensity scale
    return img * (hi - lo) + lo


# scaling statistics must come from lr alone, not hr - at inference on the real iqt data
# only the lr acquisition exists, so hr-derived statistics wouldn't be computable there.
# uses the fixed sigma_ref degradation (not a random sample) so the scale is reproducible,
# then the same (lo, hi) is applied to both lr and hr
lr_frames_ref = np.stack([degrade_hr_to_lr(f, blur_sigma=SIGMA_REF) for f in hr_frames])
percentile_scale = compute_percentile_scale(lr_frames_ref, subject, sl)
print(f"percentile scaling (from lr, sigma_ref={SIGMA_REF}) computed for {len(percentile_scale)} subject/slice series")


percentile scaling (from lr, sigma_ref=0.6369913502160143) computed for 37 subject/slice series


In [9]:
class SpatialPairDataset(torch.utils.data.Dataset):
    def __init__(self, indices, sigma_min, sigma_max, fixed_sigma=None):
        self.indices = indices
        self.sigma_min = sigma_min
        self.sigma_max = sigma_max
        self.fixed_sigma = fixed_sigma  # val and held-out sets pass this so metrics are comparable across epochs

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]
        hr = hr_frames[idx]
        lo, hi = percentile_scale[(subject[idx], sl[idx])]

        # training samples a fresh sigma every load, val/held-out always use sigma_ref
        sigma = self.fixed_sigma if self.fixed_sigma is not None else np.random.uniform(self.sigma_min, self.sigma_max)
        lr = degrade_hr_to_lr(hr, blur_sigma=sigma)

        hr_t = torch.from_numpy(normalise(hr, lo, hi)).float().unsqueeze(0)
        lr_t = torch.from_numpy(normalise(lr, lo, hi)).float().unsqueeze(0)
        return lr_t, hr_t


In [10]:
pin_memory = DEVICE == "cuda"

train_ds = SpatialPairDataset(train_idx, SIGMA_MIN, SIGMA_MAX)
val_ds = SpatialPairDataset(val_idx, SIGMA_MIN, SIGMA_MAX, fixed_sigma=SIGMA_REF)
heldout_ds = SpatialPairDataset(heldout_idx, SIGMA_MIN, SIGMA_MAX, fixed_sigma=SIGMA_REF)  # internal synthetic test set - not used below

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=pin_memory)
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin_memory)
heldout_loader = torch.utils.data.DataLoader(heldout_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin_memory)

In [11]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        # groupnorm (not batchnorm) - stays stable on the 4-image batch the local test
        # overfits, unlike batch statistics computed over so few samples
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.GroupNorm(min(8, out_ch), out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.GroupNorm(min(8, out_ch), out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class UNet(nn.Module):
    # 80x80 input, four downsampling levels (80 is divisible by 16): 80/40/20/10/5
    def __init__(self, channels):
        super().__init__()
        c1, c2, c3, c4 = channels
        self.pool = nn.MaxPool2d(2)

        self.enc1 = ConvBlock(1, c1)
        self.enc2 = ConvBlock(c1, c2)
        self.enc3 = ConvBlock(c2, c3)
        self.enc4 = ConvBlock(c3, c4)
        self.bottleneck = ConvBlock(c4, c4)

        self.up4 = nn.ConvTranspose2d(c4, c4, 2, stride=2)
        self.dec4 = ConvBlock(c4 + c4, c3)
        self.up3 = nn.ConvTranspose2d(c3, c3, 2, stride=2)
        self.dec3 = ConvBlock(c3 + c3, c2)
        self.up2 = nn.ConvTranspose2d(c2, c2, 2, stride=2)
        self.dec2 = ConvBlock(c2 + c2, c1)
        self.up1 = nn.ConvTranspose2d(c1, c1, 2, stride=2)
        self.dec1 = ConvBlock(c1 + c1, c1)

        self.out_conv = nn.Conv2d(c1, 1, 1)

    def forward(self, x):
        # encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))
        
        # decoder
        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out_conv(d1)


# residual prediction: model(lr) predicts hr - lr, not hr directly
model = UNet(CHANNELS).to(DEVICE)


In [12]:
# evaluation helpers

def psnr_batch(pred, target):
    pred_np, target_np = pred.detach().cpu().numpy(), target.detach().cpu().numpy()
    return float(np.mean([peak_signal_noise_ratio(target_np[i, 0], pred_np[i, 0], data_range=1.0) for i in range(pred_np.shape[0])]))


def ssim_batch(pred, target):
    pred_np, target_np = pred.detach().cpu().numpy(), target.detach().cpu().numpy()
    return float(np.mean([structural_similarity(target_np[i, 0], pred_np[i, 0], data_range=1.0) for i in range(pred_np.shape[0])]))


In [13]:
def run_train_epoch(model, loader, optimizer):
    model.train()
    total_loss, n = 0.0, 0
    for lr, hr in loader:
        lr, hr = lr.to(DEVICE), hr.to(DEVICE)
        pred_residual = model(lr)
        loss = F.l1_loss(pred_residual, hr - lr)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * lr.size(0)
        n += lr.size(0)
    return total_loss / n


def evaluate(model, loader):
    # also reports the degraded-input (lr vs hr) baseline so it's clear whether the
    # unet actually improves on the synthetic lr input, not just its absolute score
    model.eval()
    total = {"loss": 0.0, "psnr": 0.0, "ssim": 0.0, "baseline_psnr": 0.0, "baseline_ssim": 0.0}
    n = 0
    with torch.no_grad():
        for lr, hr in loader:
            lr, hr = lr.to(DEVICE), hr.to(DEVICE)
            pred_residual = model(lr)
            loss = F.l1_loss(pred_residual, hr - lr)
            prediction = (lr + pred_residual).clamp(0.0, 1.0)

            bs = lr.size(0)
            total["loss"] += loss.item() * bs
            total["psnr"] += psnr_batch(prediction, hr) * bs
            total["ssim"] += ssim_batch(prediction, hr) * bs
            total["baseline_psnr"] += psnr_batch(lr, hr) * bs
            total["baseline_ssim"] += ssim_batch(lr, hr) * bs
            n += bs
    return {k: v / n for k, v in total.items()}


def train_model(model, train_loader, val_loader, epochs, learning_rate, checkpoint_path, history_path):
    # held-out internal test subject is never passed in here - checkpoint selection only
    # ever sees train/val
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    best_val_loss = float("inf")
    history = []

    for epoch in range(epochs):
        epoch_start = time.time()
        train_loss = run_train_epoch(model, train_loader, optimizer)
        val_metrics = evaluate(model, val_loader)
        epoch_minutes = (time.time() - epoch_start) / 60

        print(f"epoch {epoch + 1}/{epochs}  train_l1={train_loss:.4f}  val_l1={val_metrics['loss']:.4f}  "
              f"val_psnr={val_metrics['psnr']:.2f}  val_ssim={val_metrics['ssim']:.4f}  "
              f"baseline_psnr={val_metrics['baseline_psnr']:.2f}  baseline_ssim={val_metrics['baseline_ssim']:.4f}  "
              f"epoch_time={epoch_minutes:.1f}min")
        if epoch_minutes > MAX_EPOCH_MINUTES:
            print(f"  warning: epoch exceeded MAX_EPOCH_MINUTES ({MAX_EPOCH_MINUTES})")

        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            torch.save(model.state_dict(), checkpoint_path)

        history.append({"epoch": epoch + 1, "train_loss": train_loss, "epoch_minutes": epoch_minutes, **val_metrics})

    pd.DataFrame(history).to_csv(history_path, index=False)
    return history


In [14]:
# local test - quick check that the training code runs correctly before submitting to the gpu queue
# pinned to cpu regardless of device so it runs anywhere, overfits a small
# fixed batch: a model with working gradients should easily memorise 4 frames, one that
# can't has a bug somewhere in the forward/backward path

torch.manual_seed(SEED)

test_idx = train_idx[:4]
test_ds = SpatialPairDataset(test_idx, SIGMA_MIN, SIGMA_MAX, fixed_sigma=SIGMA_REF)
test_loader = torch.utils.data.DataLoader(test_ds, batch_size=4, shuffle=False, num_workers=0)

test_model = UNet(CHANNELS).to("cpu")
test_optimizer = torch.optim.Adam(test_model.parameters(), lr=LEARNING_RATE)

lr_batch, hr_batch = next(iter(test_loader))  # same fixed 4 frames reused every iteration below

test_losses = []
last_pred_shape, last_hr_shape = None, None
for _ in range(50):
    pred_residual = test_model(lr_batch)
    loss = F.l1_loss(pred_residual, hr_batch - lr_batch)

    test_optimizer.zero_grad()
    loss.backward()
    test_optimizer.step()

    test_losses.append(loss.item())
    last_pred_shape, last_hr_shape = pred_residual.shape, hr_batch.shape

assert all(np.isfinite(test_losses)), "local test produced a non-finite loss"
assert test_losses[-1] < test_losses[0] / 10, "local test loss did not drop by at least 10x"
assert last_pred_shape == last_hr_shape, "model output shape does not match hr target shape"
test_passed = True

RUN_TRAINING = False  # flip to true to actually start training - keeps a top-to-bottom run of this notebook safe by default

if RUN_TRAINING:
    history = train_model(model, train_loader, val_loader, EPOCHS, LEARNING_RATE, CHECKPOINT_PATH, HISTORY_PATH)
    print(f"saved checkpoint to {CHECKPOINT_PATH} and history to {HISTORY_PATH}")


In [15]:
n_params = sum(p.numel() for p in UNet(CHANNELS).parameters() if p.requires_grad)

print("local test losses:", test_losses)
print("local test passed:", test_passed)
print("resolved device:", DEVICE)
print("total trainable parameters:", n_params)
print("train subjects:", train_subjects)
print("val subject:", VAL_SUBJECT)
print("internal synthetic test subject:", TEST_SUBJECT)
print("train frames:", len(train_idx))
print("val frames:", len(val_idx))
print("internal test frames:", len(heldout_idx))
print("total included frame count:", len(hr_frames))
print("echo(es) used:", echoes_used)
print("model input shape: (batch, 1, 80, 80)")
print("SIGMA_REF:", SIGMA_REF, " training sigma range:", (SIGMA_MIN, SIGMA_MAX))


local test losses: [0.3953377604484558, 0.24150283634662628, 0.1509387046098709, 0.13324841856956482, 0.11996486037969589, 0.09339621663093567, 0.07287958264350891, 0.0666334331035614, 0.061118438839912415, 0.05314009636640549, 0.04555640369653702, 0.041564054787158966, 0.04295147582888603, 0.04154140502214432, 0.037250638008117676, 0.034426189959049225, 0.03411410376429558, 0.03430230915546417, 0.03364439681172371, 0.0322444811463356, 0.030781514942646027, 0.03036496788263321, 0.03081062249839306, 0.030206002295017242, 0.029290199279785156, 0.02906828001141548, 0.028791768476366997, 0.028560465201735497, 0.02841244451701641, 0.028024496510624886, 0.0275403019040823, 0.02750224620103836, 0.02752203494310379, 0.027120281010866165, 0.026703661307692528, 0.026623735204339027, 0.026579897850751877, 0.026458825916051865, 0.026249945163726807, 0.026036690920591354, 0.026006074622273445, 0.025993842631578445, 0.025806359946727753, 0.02566971257328987, 0.02559824474155903, 0.02555084228515625,

In [16]:
RUN_TRAINING = False  # flip to true to actually start training - keeps a top-to-bottom run of this notebook safe by default

if RUN_TRAINING:
    history = train_model(model, train_loader, val_loader, EPOCHS, LEARNING_RATE, CHECKPOINT_PATH, HISTORY_PATH)
    print(f"saved checkpoint to {CHECKPOINT_PATH} and history to {HISTORY_PATH}")
